# Phase 3B — Controlled customer-grouped model tuning in Google Colab

This notebook tunes three model families using only development customers. Preprocessing is fitted separately inside every grouped cross-validation fold. Validation is used once after tuning, and final test remains sealed.

**Safety:** no Kaggle `test.csv`, SMOTE, GPU library, deep learning, nested CV, model persistence, or final-test transformation/evaluation.

## 1. Mount Drive and configure the run

**What:** mount Drive and define all paths, versions, seed, and class order. **Why:** one visible configuration makes the run auditable. **Expected:** Drive mounts and the configuration prints. **Check:** the path must point to labeled `train.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/Credit-Scoring-Model/data/raw/kaggle_credit_score/train.csv')
EXPECTED_SHA256 = 'D2EBCC056A64C48710B1AEB96777155835D372D7AD202529F64666011D214DA0'
REPOSITORY_URL = 'https://github.com/MaryamCodeHub/Credit-Scoring-Model.git'
REPOSITORY_DIR = Path('/content/Credit-Scoring-Model')
BRANCH = 'model-improvement-v2'
RANDOM_STATE = 42
CLASS_ORDER = ['Poor', 'Standard', 'Good']
print(DATASET_PATH, BRANCH, CLASS_ORDER, sep='\n')

## 2. Verify `train.csv` before reading

**What:** check existence and SHA-256 using standard-library code. **Why:** tuning the wrong file invalidates every result. **Expected:** a verification message. **Check:** stop if this cell raises; Kaggle `test.csv` is neither requested nor used.

In [ ]:
import hashlib

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Labeled train.csv not found at {DATASET_PATH}.')
digest = hashlib.sha256()
with DATASET_PATH.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        digest.update(chunk)
actual_hash = digest.hexdigest().upper()
if actual_hash != EXPECTED_SHA256:
    raise RuntimeError(f'Wrong train.csv: expected {EXPECTED_SHA256}, got {actual_hash}.')
print('Labeled train.csv SHA-256 verified.')

## 3. Clone or fast-forward the reviewed branch

**What:** clone the repository or update an existing Colab clone with `--ff-only`, then explicitly checkout the branch. **Why:** tuning must use reviewed code. **Expected:** branch and commit. **Check:** branch must equal `model-improvement-v2`.

In [ ]:
import subprocess

def run_command(arguments, cwd=None):
    return subprocess.run(arguments, cwd=cwd, check=True, text=True, capture_output=True)

if (REPOSITORY_DIR / '.git').is_dir():
    run_command(['git', 'fetch', 'origin', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'checkout', BRANCH], cwd=REPOSITORY_DIR)
    run_command(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPOSITORY_DIR)
else:
    run_command(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY_URL, str(REPOSITORY_DIR)])
active_branch = run_command(['git', 'branch', '--show-current'], cwd=REPOSITORY_DIR).stdout.strip()
commit_hash = run_command(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR).stdout.strip()
if active_branch != BRANCH:
    raise RuntimeError(f'Expected {BRANCH}, found {active_branch}.')
print(f'Branch: {active_branch}\nCommit: {commit_hash}')

## 4. Idempotent dependency setup and required restart

**What:** inspect package metadata without importing NumPy/pandas/SciPy/scikit-learn. If versions differ, reinstall the complete binary stack and deliberately restart Colab. **Why:** mixing old in-memory NumPy with new compiled packages causes mixed-version ImportErrors. **Expected:** either a deliberate disconnect or `versions already match`. **Check:** after a restart, rerun from cell 1; the second pass must continue without another restart.

In [ ]:
import importlib.metadata as metadata
import os
import signal
import sys

REQUIRED_VERSIONS = {
    'numpy': '2.4.4', 'pandas': '3.0.2', 'scipy': '1.17.1',
    'scikit-learn': '1.8.0', 'matplotlib': '3.10.8',
}
installed = {}
for name in REQUIRED_VERSIONS:
    try:
        installed[name] = metadata.version(name)
    except metadata.PackageNotFoundError:
        installed[name] = None
changed = any(installed[name] != version for name, version in REQUIRED_VERSIONS.items())
if changed:
    specs = [f'{name}=={version}' for name, version in REQUIRED_VERSIONS.items()]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--force-reinstall', *specs])
    print('Dependencies changed. Colab will restart; reconnect and rerun from cell 1.', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print('Required dependency versions already match; no restart needed.')

## 5. Import tools after dependency setup

**What:** import data, plotting, grouped-CV, pipeline, search, model, and metric tools. **Why:** binary libraries are imported only after the version gate. **Expected:** version summary. **Check:** warnings are not suppressed.

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score, make_scorer, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

os.chdir(REPOSITORY_DIR)
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from src.data_cleaning import clean_credit_data
from src.data_splitting import split_by_customer
from src.model_preprocessing import FORBIDDEN_COLUMNS, QUARANTINED_COLUMNS, build_model_preprocessor, separate_features_target_groups
print(f'NumPy {np.__version__}; pandas {pd.__version__}; scikit-learn {sklearn.__version__}')

## 6. Load labeled data, split by customer, and seal final test

**What:** load audited `train.csv` and call `split_by_customer()`. **Why:** the outer validation set must be separate from tuning, and every customer's months must stay together. **Expected:** development and validation row counts plus a sealed message. **Check:** overlap assertions pass; final-test features are not summarized, cleaned, transformed, predicted, or evaluated.

In [ ]:
raw_train = pd.read_csv(DATASET_PATH, low_memory=False)
partitions = split_by_customer(raw_train, random_state=RANDOM_STATE)
development_customers = set(partitions.development_train['Customer_ID'])
validation_customers = set(partitions.validation['Customer_ID'])
sealed_customers = set(partitions.final_test['Customer_ID'])
assert development_customers.isdisjoint(validation_customers)
assert development_customers.isdisjoint(sealed_customers)
assert validation_customers.isdisjoint(sealed_customers)
print(f'Development rows: {len(partitions.development_train):,}')
print(f'Validation rows: {len(partitions.validation):,}')
print(f'Final-test rows: {len(partitions.final_test):,}')
print(f'Final-test customers: {len(sealed_customers):,}')
print('final_test was partitioned and counted only; it is now sealed.')
del development_customers, validation_customers, sealed_customers

## 7. Clean and separate development and validation only

**What:** apply deterministic cleaning independently, then separate approved X, y, and Customer_ID groups. **Why:** identifiers, PII, target, and quarantined fields cannot enter modeling. **Expected:** input shapes and a safety confirmation. **Check:** final test is not referenced in this cell.

In [ ]:
clean_development = clean_credit_data(partitions.development_train)
clean_validation = clean_credit_data(partitions.validation)
development = separate_features_target_groups(clean_development)
validation = separate_features_target_groups(clean_validation)
unsafe = set(FORBIDDEN_COLUMNS) | set(QUARANTINED_COLUMNS)
assert unsafe.isdisjoint(development.X.columns)
assert unsafe.isdisjoint(validation.X.columns)
X_development, y_development, groups_development = development.X, development.y, development.groups
X_validation, y_validation = validation.X, validation.y
print(f'Development input: {X_development.shape}; validation input: {X_validation.shape}')
print('Approved feature contract confirmed; final_test remains untouched.')

## 8. Parameters, hyperparameters, and grouped CV

A **parameter** is learned from data, such as a logistic coefficient. A **hyperparameter** is chosen before fitting, such as tree depth or Logistic Regression C. Tuning compares a small set of hyperparameters.

**Why grouped CV:** ordinary folds could place different months of one customer on both sides. `StratifiedGroupKFold` keeps customers intact while approximately balancing classes. **Why preprocessing is inside each pipeline:** every fold must learn medians and categories from its own training customers, never from its held-out customers.

**Expected:** three folds with zero customer overlap. **Check:** all assertions pass. Validation is not part of these folds.

In [ ]:
grouped_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
for fold_number, (train_positions, held_out_positions) in enumerate(
    grouped_cv.split(X_development, y_development, groups_development), start=1
):
    train_groups = set(groups_development.iloc[train_positions])
    held_out_groups = set(groups_development.iloc[held_out_positions])
    assert train_groups.isdisjoint(held_out_groups)
    print(f'Fold {fold_number}: train customers={len(train_groups):,}, held-out customers={len(held_out_groups):,}')
macro_f1_scorer = make_scorer(f1_score, labels=CLASS_ORDER, average='macro', pos_label=None, zero_division=0)

## 9. Build pipelines and controlled searches

`GridSearchCV` evaluates every listed combination; it suits the two small grids. `RandomizedSearchCV` samples a fixed number of combinations; it limits the larger forest space to eight. Both use grouped CV and Macro F1.

Scaled Logistic Regression places StandardScaler after approved preprocessing and uses `max_iter=3000`; convergence warnings remain visible. Decision Tree uses seed 42. Random Forest uses one worker internally while the search uses two, limiting concurrent workers to two. **Expected:** three search objects. **Check:** preprocessing is the first step in every pipeline and no development-wide preprocessing occurs beforehand.

In [ ]:
logistic_pipeline = Pipeline([
    ('preprocessing', build_model_preprocessor()),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
tree_pipeline = Pipeline([
    ('preprocessing', build_model_preprocessor()),
    ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
forest_pipeline = Pipeline([
    ('preprocessing', build_model_preprocessor()),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)),
])

searches = {
    'Scaled Logistic Regression': GridSearchCV(
        logistic_pipeline,
        {'classifier__C': [0.1, 1.0, 10.0], 'classifier__class_weight': [None, 'balanced']},
        scoring=macro_f1_scorer, cv=grouped_cv, n_jobs=2, refit=True, return_train_score=True,
    ),
    'Decision Tree': GridSearchCV(
        tree_pipeline,
        {
            'classifier__max_depth': [6, 8, 10, 12],
            'classifier__min_samples_leaf': [25, 50, 100],
            'classifier__class_weight': [None, 'balanced'],
        },
        scoring=macro_f1_scorer, cv=grouped_cv, n_jobs=2, refit=True, return_train_score=True,
    ),
    'Random Forest': RandomizedSearchCV(
        forest_pipeline,
        {
            'classifier__n_estimators': [100, 150, 200],
            'classifier__max_depth': [10, 12, 16, None],
            'classifier__min_samples_leaf': [5, 10, 20],
            'classifier__max_features': ['sqrt', 0.7],
            'classifier__class_weight': ['balanced', 'balanced_subsample'],
        },
        n_iter=8, scoring=macro_f1_scorer, cv=grouped_cv, n_jobs=2,
        random_state=RANDOM_STATE, refit=True, return_train_score=True,
    ),
}
print('Controlled searches:', list(searches))

## 10. Tune on development groups only

**What:** pass Customer_ID groups to each search. **Why:** preprocessing and model fitting repeat independently within each CV training fold. `refit=True` then fits the selected full pipeline once on complete development. **Expected:** best parameters, CV Macro F1 mean/std, and approximate time. **Check:** validation is absent from `fit()`.

In [ ]:
tuning_summaries = {}
for model_name, search in searches.items():
    started = time.perf_counter()
    search.fit(X_development, y_development, groups=groups_development)
    tuning_seconds = time.perf_counter() - started
    best_index = search.best_index_
    summary = {
        'Best Parameters': search.best_params_,
        'CV Mean Macro F1': search.cv_results_['mean_test_score'][best_index],
        'CV Std Macro F1': search.cv_results_['std_test_score'][best_index],
        'Tuning Seconds': tuning_seconds,
        'Refit Seconds': search.refit_time_,
    }
    tuning_summaries[model_name] = summary
    print(f'\n{model_name}\n{summary}')

## 11. Evaluate selected pipelines once on validation

**What:** calculate requested metrics, per-class reports, confusion matrices, development score, gap, and prediction time. **Why:** validation is an outer, unseen-customer check after search spaces are fixed. **Expected:** a sorted comparison table. **Check:** prioritize Macro F1 and Poor recall, then examine Standard/Good and reject suspiciously large gaps.

In [ ]:
rows, reports, matrices = [], {}, {}
for model_name, search in searches.items():
    selected = search.best_estimator_
    development_predictions = selected.predict(X_development)
    started = time.perf_counter()
    validation_predictions = selected.predict(X_validation)
    prediction_seconds = time.perf_counter() - started
    train_macro = f1_score(y_development, development_predictions, labels=CLASS_ORDER, average='macro', zero_division=0)
    validation_macro = f1_score(y_validation, validation_predictions, labels=CLASS_ORDER, average='macro', zero_division=0)
    report = classification_report(y_validation, validation_predictions, labels=CLASS_ORDER, target_names=CLASS_ORDER, output_dict=True, zero_division=0)
    summary = tuning_summaries[model_name]
    rows.append({
        'Model': model_name,
        'Validation Macro F1': validation_macro,
        'Development Macro F1': train_macro,
        'Train-Validation Gap': train_macro - validation_macro,
        'Accuracy': accuracy_score(y_validation, validation_predictions),
        'Balanced Accuracy': balanced_accuracy_score(y_validation, validation_predictions),
        'Weighted F1': f1_score(y_validation, validation_predictions, labels=CLASS_ORDER, average='weighted', zero_division=0),
        'Macro Precision': precision_score(y_validation, validation_predictions, labels=CLASS_ORDER, average='macro', zero_division=0),
        'Macro Recall': recall_score(y_validation, validation_predictions, labels=CLASS_ORDER, average='macro', zero_division=0),
        'Poor Recall': report['Poor']['recall'],
        'CV Mean Macro F1': summary['CV Mean Macro F1'],
        'CV Std Macro F1': summary['CV Std Macro F1'],
        'Refit Seconds': summary['Refit Seconds'],
        'Prediction Seconds': prediction_seconds,
    })
    reports[model_name] = report
    matrices[model_name] = confusion_matrix(y_validation, validation_predictions, labels=CLASS_ORDER)

comparison = pd.DataFrame(rows).sort_values('Validation Macro F1', ascending=False).reset_index(drop=True)
display(comparison.style.format(precision=4))

## 12. Compare with Phase 3A baselines

**What:** paste the four validation Macro F1 values from your completed Phase 3A run. **Why:** notebook outputs and models are intentionally not persisted, so explicit transfer keeps provenance visible. **Expected:** one combined table. **Check:** fill every `None` before continuing; do not alter Phase 3B search spaces after seeing validation.

In [ ]:
PHASE_3A_VALIDATION_MACRO_F1 = {
    'DummyClassifier': None,
    'LogisticRegression': None,
    'DecisionTreeClassifier': None,
    'RandomForestClassifier': None,
}
if any(value is None for value in PHASE_3A_VALIDATION_MACRO_F1.values()):
    raise ValueError('Paste all four Phase 3A validation Macro F1 values before comparison.')
baseline_rows = [
    {'Phase': '3A baseline', 'Model': name, 'Validation Macro F1': score}
    for name, score in PHASE_3A_VALIDATION_MACRO_F1.items()
]
tuned_rows = [
    {'Phase': '3B tuned', 'Model': row['Model'], 'Validation Macro F1': row['Validation Macro F1']}
    for _, row in comparison.iterrows()
]
display(pd.DataFrame(baseline_rows + tuned_rows).sort_values('Validation Macro F1', ascending=False))

## 13. Per-class reports and confusion matrices

**What:** inspect Poor, Standard, and Good precision/recall/F1 and visualize errors. **Why:** one aggregate metric can hide missed Poor-risk customers. **Expected:** three reports and matrices. **Check:** examine Poor recall first, but do not ignore the other classes.

In [ ]:
for model_name in searches:
    print(f'\n{model_name}')
    display(pd.DataFrame(reports[model_name]).T.loc[CLASS_ORDER, ['precision', 'recall', 'f1-score', 'support']])
figure, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, model_name in zip(axes, searches):
    ConfusionMatrixDisplay(matrices[model_name], display_labels=CLASS_ORDER).plot(ax=axis, cmap='Blues', colorbar=False, values_format='d')
    axis.set_title(model_name)
figure.tight_layout()
plt.show()

## 14. Decision guidance—not an automatic winner

**What:** summarize the leading validation model, Poor recall, CV stability, and train–validation gap. **Why:** the primary metric is validation Macro F1, but material Poor recall, unstable CV, or a large gap can outweigh a small score advantage. **Expected:** a compact review table. **Check:** do not declare production readiness or tune again after seeing validation; final test remains sealed.

In [ ]:
decision_columns = ['Model', 'Validation Macro F1', 'Poor Recall', 'CV Mean Macro F1', 'CV Std Macro F1', 'Train-Validation Gap']
display(comparison[decision_columns])
leader = comparison.iloc[0]
print(f"Highest validation Macro F1: {leader['Model']} ({leader['Validation Macro F1']:.4f})")
print('Review Poor recall, Standard/Good reports, CV standard deviation, timing, and gap before deciding.')
print('A large positive train-validation gap is a rejection warning, not evidence of strength.')

## 15. Final safety assertion

**What:** verify that final test was only partitioned and counted: it was not cleaned; its features, target, or groups were not separated; and it was not transformed, predicted, or evaluated. **Why:** final test is reserved for a later locked evaluation. **Expected:** safety confirmations. **Check:** stop if an assertion fails.

In [ ]:
assert 'clean_final_test' not in globals()
assert 'final_test_inputs' not in globals()
assert '_sealed_final_inputs' not in globals()
assert 'X_final_test' not in globals()
assert 'X_final_test_transformed' not in globals()
assert 'final_test_predictions' not in globals()
print('Confirmed: final_test was partitioned and counted only.')
print('Confirmed: final_test was not cleaned or separated into features, target, or groups.')
print('Confirmed: final_test was not cleaned, transformed, predicted, or evaluated.')
print('Confirmed: no model, search object, or transformed dataset was persisted.')
print('Phase 3B results are validation evidence only—not production readiness.')